# GATv2 Architecture-Level Check on GLOBEM
## Component 2 — Anxiety Vulnerability Mapping (R26-DS-012)
### Same GATv2 model class as `digital_phenotyping_v14_polished.ipynb`, adapted graph construction for GLOBEM's data structure

---
**What changed and why (read before quoting results anywhere):**

Your StudentLife pipeline builds one graph per participant where **nodes =
contextual states** (`LOC_x__TIME__ACTIVITY`), derived by DBSCAN-clustering
raw GPS pings into stay-points. GLOBEM's public release does not include
raw GPS pings — only RAPIDS-derived daily aggregate features — so that
specific node definition cannot be reconstructed here (confirmed in the
prior exploration notebook).

**This notebook keeps the GATv2 architecture and the feature-window \u2192
label-window no-leakage discipline, but changes what a node IS:**

| | StudentLife (`v14_polished`) | This notebook (GLOBEM) |
|---|---|---|
| Node | Contextual state (location \u00d7 time \u00d7 activity) | **One calendar day** |
| Node features | visit_count, typical_hour, ema_stress, etc. (10-dim, engineered) | GLOBEM's own `:allday` RAPIDS features + that day's mean EMA (~83-dim, off-the-shelf) |
| Edge | Sequential state-to-state transition (gap < 4h) | Sequential day-to-day (chronological) |
| Graph scope | Whole 35-day feature window, 1 graph/participant | Trailing 14-day window before each weekly assessment, 1 graph/participant-week |
| Label | PSS score (post-study), 1 label/participant | `anx_weekly_subscale`, 1 label/participant-week |
| Model | GATv2Conv \u00d7 N + pooling + heads | GATv2Conv \u00d7 2 + mean/max pooling + MLP head |

This is a genuine **architecture-level** generalization check (same GNN
class reasoning over a graph, same train/test temporal-leakage discipline)
but **not an identical replication** \u2014 the node semantics differ because
the underlying data differs. State that distinction explicitly wherever
you report these numbers; don't call it "the same graph."

Reported with the same 4-number honest discipline as Component 2: main
(location+EMA) AUC, behavioral-only (no EMA) AUC, permutation AUC, honest
range \u2014 now for **both** the GATv2 model and a tabular RandomForest
baseline, so you can see whether the graph structure adds anything over
flat features.


In [ ]:
# ============================================================
# CELL 1: Install, mount Drive, imports, seeds
# ============================================================
!pip install torch torch_geometric -q

from google.colab import drive
drive.mount('/content/drive')

import os, time, warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device.type == 'cpu':
    print("\u26a0 No GPU detected \u2014 Runtime > Change runtime type > GPU will speed this up a lot.")
print("\u2705 Libraries loaded")


In [ ]:
# ============================================================
# CELL 2: Configuration
# ============================================================
GLOBEM_ROOT = "/content/drive/MyDrive/Anxiety/globem-dataset-multi-year-datasets-for-longitudinal-human-behavior-modeling-generalization-1.1/globem-dataset-multi-year-datasets-for-longitudinal-human-behavior-modeling-generalization-1.1/"
YEARS = ["INS-W_1", "INS-W_2", "INS-W_3", "INS-W_4"]
OUTPUT_DIR = "/content/drive/MyDrive/Anxiety/globem_gatv2_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Graph construction
WINDOW_DAYS = 14     # trailing days used to build each graph (mirrors your
                      # feature-window concept, compressed to weekly cadence)
MIN_NODES   = 3       # StudentLife used the same >=3 node minimum

# Node features: GLOBEM's own per-day (':allday') continuous RAPIDS columns.
# '_dis:' variants (categorical low/med/high) are excluded, same choice as
# the tabular exploration notebook.
NODE_FEATURE_SUFFIX = ":allday"

# GATv2 architecture
HIDDEN  = 32
HEADS   = 4
DROPOUT = 0.3

# Training / evaluation
N_SPLITS  = 5
N_REPEATS = 3     # reduce to 1-2 for a quick check, increase for final numbers
EPOCHS    = 30
BATCH_SIZE = 32
LR = 0.001
WEIGHT_DECAY = 5e-4

print("\u2705 Configuration set")


In [ ]:
# ============================================================
# CELL 3: Per-year data loading (same loader as the tabular notebook)
# ============================================================
def load_year(year):
    base = os.path.join(GLOBEM_ROOT, year)
    loc_path = os.path.join(base, "FeatureData", "location.csv")
    ema_path = os.path.join(base, "SurveyData", "ema.csv")
    dep_path = os.path.join(base, "SurveyData", "dep_weekly.csv")

    if not (os.path.exists(loc_path) and os.path.exists(dep_path)):
        print(f"  \u26a0 {year}: missing location.csv or dep_weekly.csv \u2014 skipped")
        return None

    loc = pd.read_csv(loc_path)
    loc["date"] = pd.to_datetime(loc["date"])

    dep = pd.read_csv(dep_path)
    dep["date"] = pd.to_datetime(dep["date"])
    dep = dep.dropna(subset=["anx_weekly_subscale"]).copy()
    dep["label"] = dep["anx_weekly_subscale"].astype(bool).astype(int)

    if os.path.exists(ema_path):
        ema = pd.read_csv(ema_path)
        ema["date"] = pd.to_datetime(ema["date"])
    else:
        ema = pd.DataFrame(columns=["pid", "date", "negative_affect_EMA"])

    return {"location": loc, "ema": ema, "dep_weekly": dep}

print("Loading GLOBEM years...")
raw_data = {}
for yr in YEARS:
    result = load_year(yr)
    if result is not None:
        raw_data[yr] = result
        print(f"  \u2705 {yr}: {result['location'].pid.nunique()} participants, "
              f"{len(result['dep_weekly'])} labelled weeks")

if not raw_data:
    raise RuntimeError("No GLOBEM years found \u2014 check GLOBEM_ROOT in CELL 2.")

# Common node-feature columns across all loaded years
common_node_cols = None
for yr, d in raw_data.items():
    cols = set(c for c in d["location"].columns
               if c.endswith(NODE_FEATURE_SUFFIX) and "_dis:" not in c)
    common_node_cols = cols if common_node_cols is None else (common_node_cols & cols)
NODE_COLS = sorted(common_node_cols)
print(f"\n\u2705 Loaded {len(raw_data)} year(s): {list(raw_data.keys())}")
print(f"\u2705 {len(NODE_COLS)} common daily node-feature columns (+1 for EMA = {len(NODE_COLS)+1}-dim node vectors)")


In [ ]:
# ============================================================
# CELL 4: Graph construction \u2014 one graph per participant-week
# ============================================================
# Node = one calendar day in the trailing WINDOW_DAYS before the
# assessment date. Edge = sequential day-to-day (chronological order),
# mirroring the sequential transition edges in your StudentLife graphs.
# Graph label = that week's anx_weekly_subscale.
#
# This preserves the feature-window -> label-window split discipline:
# only days STRICTLY BEFORE the assessment date are used as nodes.

def build_graphs_for_year(year, data, include_ema=True):
    loc, ema, dep = data["location"], data["ema"], data["dep_weekly"]
    ema_daily = ema.groupby(["pid", "date"])["negative_affect_EMA"].mean().rename("ema_day")
    graphs = []
    for row in dep.itertuples():
        pid, end_date, label = row.pid, row.date, row.label
        start_date = end_date - pd.Timedelta(days=WINDOW_DAYS)
        days = pd.date_range(start_date + pd.Timedelta(days=1), end_date, freq="D")
        sub = loc[(loc.pid == pid) & (loc.date.isin(days))].sort_values("date")
        if len(sub) < MIN_NODES:
            continue

        feats = sub[NODE_COLS].values.astype(np.float32)
        if include_ema:
            ema_vals = sub["date"].map(lambda d: ema_daily.get((pid, d), np.nan)).values.astype(np.float32)
            node_x = np.concatenate([feats, ema_vals.reshape(-1, 1)], axis=1)
        else:
            node_x = feats

        n = len(sub)
        src = list(range(n - 1)); dst = list(range(1, n))
        edge_index = torch.tensor([src + dst, dst + src], dtype=torch.long)  # bidirectional

        g = Data(x=torch.tensor(node_x, dtype=torch.float),
                  edge_index=edge_index,
                  y=torch.tensor([label], dtype=torch.float))
        g.pid = pid
        g.year = year
        graphs.append(g)
    return graphs

print("Building graphs per year...")
graphs_by_year_ema    = {}   # location + EMA node features
graphs_by_year_beh    = {}   # location-only (no EMA) ablation
for yr, data in raw_data.items():
    graphs_by_year_ema[yr] = build_graphs_for_year(yr, data, include_ema=True)
    graphs_by_year_beh[yr] = build_graphs_for_year(yr, data, include_ema=False)
    n = len(graphs_by_year_ema[yr])
    n_pos = np.mean([g.y.item() for g in graphs_by_year_ema[yr]]) if n else float('nan')
    print(f"  {yr}: {n} graphs, {n_pos:.1%} positive-week")

all_graphs_ema = sum(graphs_by_year_ema.values(), [])
all_graphs_beh = sum(graphs_by_year_beh.values(), [])
print(f"\n\u2705 {len(all_graphs_ema)} total graphs across {len(graphs_by_year_ema)} year(s)")


In [ ]:
# ============================================================
# CELL 5: Model \u2014 GATv2 (same layer choices as v14: GATv2Conv +
# mean/max global pooling), plus per-fold imputer/scaler
# ============================================================
class GraphImputer:
    # Median-impute missing node features, fit on TRAIN graphs only.
    def fit(self, graphs):
        all_x = torch.cat([g.x for g in graphs], dim=0)
        med = torch.nan_to_num(all_x, nan=float('nan'))
        self.median = torch.nan_to_num(med.nanmedian(dim=0).values, nan=0.0)
        return self
    def transform(self, graphs):
        out = []
        for g in graphs:
            x = g.x.clone()
            mask = torch.isnan(x)
            x[mask] = self.median.unsqueeze(0).expand_as(x)[mask]
            g2 = g.clone(); g2.x = x; out.append(g2)
        return out

class GraphScaler:
    # Z-score node features, fit on TRAIN graphs only. Essential \u2014 raw
    # RAPIDS features (meters, minutes, counts) span very different scales
    # and blow up GATv2 training loss without this.
    def fit(self, graphs):
        all_x = torch.cat([g.x for g in graphs], dim=0)
        self.mean = all_x.mean(dim=0)
        self.std  = all_x.std(dim=0)
        self.std[self.std < 1e-6] = 1.0
        return self
    def transform(self, graphs):
        out = []
        for g in graphs:
            g2 = g.clone()
            g2.x = (g.x - self.mean.unsqueeze(0)) / self.std.unsqueeze(0)
            out.append(g2)
        return out

class GATv2Classifier(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, heads=HEADS, dropout=DROPOUT):
        super().__init__()
        self.conv1 = GATv2Conv(in_dim, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATv2Conv(hidden * heads, hidden, heads=1, dropout=dropout)
        self.lin1  = nn.Linear(hidden * 2, hidden)
        self.lin2  = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv2(x, edge_index))
        g = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        g = F.relu(self.lin1(g))
        g = F.dropout(g, p=self.dropout, training=self.training)
        return self.lin2(g).squeeze(-1)

print("\u2705 Model classes ready")


In [ ]:
# ============================================================
# CELL 6: Training / evaluation helpers
# ============================================================
def run_fold(train_graphs, test_graphs, epochs=EPOCHS):
    imp = GraphImputer().fit(train_graphs)
    train_graphs = imp.transform(train_graphs)
    test_graphs  = imp.transform(test_graphs)
    scaler = GraphScaler().fit(train_graphs)
    train_graphs = scaler.transform(train_graphs)
    test_graphs  = scaler.transform(test_graphs)

    in_dim = train_graphs[0].x.shape[1]
    model = GATv2Classifier(in_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_graphs, batch_size=64)

    y_train = np.array([g.y.item() for g in train_graphs])
    pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)]).to(device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = loss_fn(out, batch.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch))
            preds.extend(out.cpu().tolist())
            trues.extend(batch.y.cpu().tolist())
    return roc_auc_score(trues, preds)

def cv_evaluate(graphs, label, n_splits=N_SPLITS, n_repeats=N_REPEATS, epochs=EPOCHS, verbose=True):
    y = np.array([g.y.item() for g in graphs])
    groups = np.array([g.pid for g in graphs])
    aucs = []
    t0 = time.time()
    for rep in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED + rep)
        for tr_idx, te_idx in sgkf.split(np.zeros(len(graphs)), y, groups):
            train_g = [graphs[i] for i in tr_idx]
            test_g  = [graphs[i] for i in te_idx]
            aucs.append(run_fold(train_g, test_g, epochs=epochs))
    if verbose:
        print(f"  {label}: AUC {np.mean(aucs):.3f} \u00b1 {np.std(aucs):.3f}  "
              f"({time.time()-t0:.0f}s, {len(aucs)} folds)")
    return np.mean(aucs), np.std(aucs)

def permute_labels(graphs, seed=SEED):
    graphs_p = [g.clone() for g in graphs]
    y = np.array([g.y.item() for g in graphs_p])
    rng = np.random.RandomState(seed)
    rng.shuffle(y)
    for g, yl in zip(graphs_p, y):
        g.y = torch.tensor([yl], dtype=torch.float)
    return graphs_p

print("\u2705 Training/evaluation helpers ready")


In [ ]:
# ============================================================
# CELL 7: WITHIN-COHORT GATv2 results \u2014 per year + pooled
# ============================================================
print("=" * 70)
print("  GATv2 WITHIN-COHORT RESULTS")
print("=" * 70)

gatv2_within = []
for yr in graphs_by_year_ema:
    print(f"\n-- {yr} ({len(graphs_by_year_ema[yr])} graphs) --")
    auc_main, _ = cv_evaluate(graphs_by_year_ema[yr], f"{yr} GATv2 location+EMA")
    auc_beh,  _ = cv_evaluate(graphs_by_year_beh[yr], f"{yr} GATv2 location-only")
    auc_perm, _ = cv_evaluate(permute_labels(graphs_by_year_ema[yr]),
                               f"{yr} GATv2 permutation", n_repeats=1)
    gatv2_within.append({"year": yr, "n_graphs": len(graphs_by_year_ema[yr]),
                          "auc_main": auc_main, "auc_behavioral_only": auc_beh,
                          "auc_permutation": auc_perm})

print(f"\n-- POOLED ({len(all_graphs_ema)} graphs) --")
pooled_main, _ = cv_evaluate(all_graphs_ema, "Pooled GATv2 location+EMA")
pooled_beh,  _ = cv_evaluate(all_graphs_beh, "Pooled GATv2 location-only")
pooled_perm, _ = cv_evaluate(permute_labels(all_graphs_ema), "Pooled GATv2 permutation", n_repeats=1)

gatv2_within_df = pd.DataFrame(gatv2_within)
gatv2_within_df.to_csv(OUTPUT_DIR + "gatv2_within_cohort_results.csv", index=False)
print("\n\u2705 GATv2 within-cohort results saved")


In [ ]:
# ============================================================
# CELL 8: Leave-one-year-out cross-cohort GATv2 validation
# ============================================================
# Same rationale as the tabular notebook: each INS-W_n year is a
# separately-recruited cohort in a different period, so train-on-N-1
# / test-on-held-out-year is the closest thing to a genuine external
# validation GLOBEM's public data supports. Only runs with 2+ years.

def run_holdout(train_graphs, test_graphs, epochs=EPOCHS):
    return run_fold(train_graphs, test_graphs, epochs=epochs)

gatv2_loyo = []
if len(graphs_by_year_ema) >= 2:
    print("=" * 70)
    print("  GATv2 LEAVE-ONE-YEAR-OUT CROSS-COHORT VALIDATION")
    print("=" * 70)
    for held_out in graphs_by_year_ema:
        for feat_set, by_year in [("location+EMA", graphs_by_year_ema),
                                    ("location-only", graphs_by_year_beh)]:
            train_g = sum([g for yr, g in by_year.items() if yr != held_out], [])
            test_g  = by_year[held_out]
            auc = run_holdout(train_g, test_g)
            print(f"  Train=all-but-{held_out}  Test={held_out}  [{feat_set}]  AUC={auc:.3f}")
            gatv2_loyo.append({"held_out_year": held_out, "feature_set": feat_set,
                                "auc": auc, "n_train": len(train_g), "n_test": len(test_g)})
else:
    print("Only 1 year loaded \u2014 leave-one-year-out needs 2+ years. "
          "Download additional INS-W_n years and re-run from CELL 3.")

gatv2_loyo_df = pd.DataFrame(gatv2_loyo)
if len(gatv2_loyo_df):
    gatv2_loyo_df.to_csv(OUTPUT_DIR + "gatv2_leave_one_year_out_results.csv", index=False)
    print("\n\u2705 GATv2 leave-one-year-out results saved")


In [ ]:
# ============================================================
# CELL 9: Tabular RandomForest baseline (for GNN-vs-flat-features comparison)
# ============================================================
# Same features (day-level, flattened to weekly means over the same
# WINDOW_DAYS window) fed to a RandomForest, so you can report whether
# the graph structure itself adds anything over flat aggregate features.

def graphs_to_tabular(graphs):
    rows = []
    for g in graphs:
        x = g.x.numpy()
        row = np.nanmean(x, axis=0)   # mean over the day-nodes in the window
        rows.append(row)
    X = np.array(rows)
    y = np.array([g.y.item() for g in graphs])
    groups = np.array([g.pid for g in graphs])
    return X, y, groups

def rf_evaluate(graphs, label, n_splits=N_SPLITS, n_repeats=N_REPEATS):
    X, y, groups = graphs_to_tabular(graphs)
    aucs = []
    for rep in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED + rep)
        for tr_idx, te_idx in sgkf.split(X, y, groups):
            imp = SimpleImputer(strategy="median")
            X_train = imp.fit_transform(X[tr_idx]); X_test = imp.transform(X[te_idx])
            clf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                          class_weight="balanced", random_state=SEED)
            clf.fit(X_train, y[tr_idx])
            proba = clf.predict_proba(X_test)[:, 1]
            aucs.append(roc_auc_score(y[te_idx], proba))
    print(f"  {label}: AUC {np.mean(aucs):.3f} \u00b1 {np.std(aucs):.3f}")
    return np.mean(aucs), np.std(aucs)

print("=" * 70)
print("  RANDOMFOREST BASELINE (pooled, same window/features, no graph structure)")
print("=" * 70)
rf_main, _ = rf_evaluate(all_graphs_ema, "RF pooled location+EMA")
rf_beh,  _ = rf_evaluate(all_graphs_beh, "RF pooled location-only")
rf_perm, _ = rf_evaluate(permute_labels(all_graphs_ema), "RF pooled permutation", n_repeats=1)


In [ ]:
# ============================================================
# CELL 10: Summary \u2014 GATv2 vs RandomForest, 4-number honest story
# ============================================================
print("=" * 70)
print("  FINAL SUMMARY \u2014 GATv2 ARCHITECTURE CHECK ON GLOBEM")
print("=" * 70)
print(f"  Years loaded  : {list(graphs_by_year_ema.keys())}")
print(f"  Total graphs  : {len(all_graphs_ema)}")
print(f"  Node dim      : {len(NODE_COLS)} location + 1 EMA = {len(NODE_COLS)+1}")
print(f"  Window        : trailing {WINDOW_DAYS} days, 1 graph per participant-week")
print()
print("  -- POOLED WITHIN-COHORT, GATv2 vs RandomForest --------------")
print(f"  {'Model':<14} {'Main (loc+EMA)':>16} {'Behavioral-only':>17} {'Permutation':>13}")
print(f"  {'GATv2':<14} {pooled_main:>16.3f} {pooled_beh:>17.3f} {pooled_perm:>13.3f}")
print(f"  {'RandomForest':<14} {rf_main:>16.3f} {rf_beh:>17.3f} {rf_perm:>13.3f}")
print()
if len(gatv2_loyo_df):
    print("  -- CROSS-COHORT (leave-one-year-out), GATv2 -------------------")
    for _, r in gatv2_loyo_df.iterrows():
        print(f"  Held out {r['held_out_year']:<10} [{r['feature_set']:<14}] AUC={r['auc']:.3f}")
print()
print("  SCOPE REMINDER: node = calendar day here, not contextual state as")
print("  in StudentLife. Same GATv2 architecture, adapted graph construction")
print("  due to GLOBEM's aggregate-feature-only public release. Report this")
print("  distinction explicitly \u2014 do not describe it as an identical replication.")
print(f"\nAll result files written to: {OUTPUT_DIR}")
